**creating catalog for healthcare project**

In [0]:
%sql
show catalogs;

In [0]:
%sql
create catalog if not exists healthcare;

In [0]:
%sql
USE CATALOG healthcare;

In [0]:
%sql
create schema if not exists bronze;

In [0]:
%sql
create schema if not exists silver;

In [0]:
%sql
create schema if not exists gold;

**Ingecting data from different source to bronze **

In [0]:
health_diagnoses_df = spark.read.table("healthcare.default.health_diagnoses")
health_encounters_df = spark.read.table("healthcare.default.health_encounters")
health_patients_df = spark.read.table("healthcare.default.health_patients")
health_treatments_df = spark.read.table("healthcare.default.health_treatments")

In [0]:
display(health_patients_df.count())

In [0]:
last_run_date = "2023-05-01"


In [0]:
from pyspark.sql.functions import col

incremental_df = health_encounters_df.filter(
    col("admit_date") > last_run_date
)


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "healthcare.bronze.health_encounters_bronze")

delta_table.alias("target").merge(
    incremental_df.alias("source"),
    "target.encounter_id = source.encounter_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

In [0]:
health_diagnoses_df.write.mode("overwrite").saveAsTable("healthcare.bronze.health_diagnoses_bronze")
health_encounters_df.write.mode("overwrite").saveAsTable("healthcare.bronze.health_encounters_bronze")
health_patients_df.write.mode("overwrite").saveAsTable("healthcare.bronze.health_patients_bronze")
health_treatments_df.write.mode("overwrite").saveAsTable("healthcare.bronze.health_treatments_bronze")

**Silver layer transformations and ingest to gold layer**

In [0]:
health_diagnoses_bronze = spark.read.table("healthcare.bronze.health_diagnoses_bronze")
health_diagnoses_bronze.createOrReplaceTempView("health_diagnoses_bronze")


health_encounters_bronze = spark.read.table("healthcare.bronze.health_encounters_bronze")
health_encounters_bronze.createOrReplaceTempView("health_encounters_bronze")


health_patients_bronze = spark.read.table("healthcare.bronze.health_patients_bronze")
health_patients_bronze.createOrReplaceTempView("health_patients_bronze")


health_treatments_bronze = spark.read.table("healthcare.bronze.health_treatments_bronze")
health_treatments_bronze.createOrReplaceTempView("health_treatments_bronze")

In [0]:
df = spark.sql ("select * from health_diagnoses_bronze limit 5")
df.display()

In [0]:
df1 = spark.sql ("select * from health_encounters_bronze limit 5")
df1.display()

In [0]:
health_encounters_bronze.write.partitionBy("department") \
    .format("delta") \
    .saveAsTable("healthcare.silver.partition_by_department")


In [0]:
dfd = spark.read.table("healthcare.silver.partition_by_department")
display(dfd)

In [0]:
df2 = spark.sql ("select * from health_patients_bronze limit 5")
df2.display()

In [0]:
df3 = spark.sql ("select * from health_treatments_bronze limit 5")
df3.display()

In [0]:
display(health_patients_bronze.count())

In [0]:
patients_clean = health_patients_bronze.dropDuplicates() \
    .filter("patient_id is NOT NULL") \
    .fillna({"age": 0})

In [0]:
patients_clean1 = health_patients_bronze.groupBy("patient_id").count().filter("count >1")

In [0]:
display(patients_clean1.count())

In [0]:
patients_clean.display()

In [0]:
from pyspark.sql.functions import to_date

encounters_clean = health_encounters_bronze.withColumn("admit_date", to_date("admit_date"))

In [0]:
from pyspark.sql.functions import date_diff

encounters_clean1 = encounters_clean.withColumn("length_of_stay", date_diff("discharge_date", "admit_date"))

In [0]:
encounters_clean1.display()

In [0]:
join_df = spark.sql("select health_patients_bronze.patient_id, health_patients_bronze.name, health_patients_bronze.age , health_patients_bronze.gender, health_patients_bronze.city, health_patients_bronze.registration_date, health_encounters_bronze.encounter_id as enocunter_encouter_id, health_encounters_bronze.patient_id as enocunter_patient_id, health_encounters_bronze.admit_date, health_encounters_bronze.discharge_date, health_encounters_bronze.department, health_diagnoses_bronze.diagnosis_id, health_diagnoses_bronze.encounter_id as diagnosis_encounter_id, health_diagnoses_bronze.diagnosis_code, health_diagnoses_bronze.diagnosis_desc, health_treatments_bronze.treatment_id, health_treatments_bronze.encounter_id as treatment_encounter_id, health_treatments_bronze.procedure_code, health_treatments_bronze.procedure_desc, health_treatments_bronze.cost from health_patients_bronze join health_encounters_bronze on health_patients_bronze.patient_id = health_encounters_bronze.patient_id join health_diagnoses_bronze on health_encounters_bronze.encounter_id = health_diagnoses_bronze.encounter_id join health_treatments_bronze on health_encounters_bronze.encounter_id = health_treatments_bronze.encounter_id")

In [0]:
join_df.display()
join_df.createOrReplaceTempView("final_table")

In [0]:
spark.sql("OPTIMIZE healthcare.gold.fact_gold_final")

In [0]:
%python
spark.sql(
    "OPTIMIZE healthcare.silver.health_data_silver ZORDER BY (patient_id)"
)


In [0]:
from pyspark.sql.functions import broadcast

df_join = health_patients_bronze.join(broadcast(health_encounters_bronze), "patient_id")


In [0]:
df_join.display()

In [0]:
join_df1 = join_df.withColumn("length_of_stay", date_diff("discharge_date", "admit_date"))

In [0]:
join_df1.display()

In [0]:
from pyspark.sql.functions import expr

join_df2 = join_df1.withColumn("risk_level", expr("""case when diagnosis_code = 'I10' then 'High' when diagnosis_code = 'I12' then 'Medium' else 'Low' END"""))

In [0]:
join_df2.display()

In [0]:
join_df3 = join_df2.withColumn("age_group", expr(""" case when age < 18 then 'Child' when age >= 18 and age <= 60 then 'Adult' else 'Senior' END """))

In [0]:
join_df3.display()

In [0]:
join_df3.display()

In [0]:
# Drop duplicate aliased columns to match existing table schema
join_df3_clean = join_df3.drop("enocunter_patient_id", "diagnosis_encounter_id", "treatment_encounter_id")

join_df3_clean.write.format("delta").mode("overwrite").saveAsTable("healthcare.silver.health_data_silver")

In [0]:
join_df3_clean.display()

**GOLD LAYER (BUSINESS KPIs — VERY IMPORTANT)**

In [0]:
gold_layer = spark.read.table("healthcare.silver.health_data_silver")

In [0]:
gold_layer.display()

In [0]:
from pyspark.sql.functions import sum

gold_revenue = gold_layer.groupBy("department") \
    .agg(sum("cost").alias("total_revenue"))

In [0]:
gold_revenue.display()

In [0]:
gold_revenue.write.format("Delta").mode("overwrite").saveAsTable("healthcare.gold.dim_revenue")

In [0]:
from pyspark.sql.functions import avg
gold_los = gold_layer.groupBy("department") \
    .agg(avg("length_of_stay").alias("avg_los"))

In [0]:
gold_los.display()

In [0]:
gold_los.write.format("Delta").mode("overwrite").saveAsTable("healthcare.gold.dim_los")

In [0]:
from pyspark.sql.functions import countDistinct

disease_count = gold_layer.groupBy("diagnosis_desc") \
    .agg(countDistinct("patient_id").alias("patient_count"))


In [0]:
disease_count.display()

In [0]:
disease_count.write.format("Delta").mode("overwrite").saveAsTable("healthcare.gold.dim_disease")

In [0]:
gold_layer.write.format("Delta").mode("overwrite").saveAsTable("healthcare.gold.fact_gold_final")